# Used Car Price Prediction: KNN

### Dataset

It is a comma separated file and there are 14 columns in the dataset.

- Location - The location in which the car is being sold or is available for purchase.
- Year - The year or edition of the model.
- KM_Driven - The total kilometers are driven in the car by the previous owner(s) in '000 KM.
- Fuel_Type - The type of fuel used by the car. (Petrol, Diesel, Electric, CNG, LPG)
- Transmission - The type of transmission used by the car. (Automatic / Manual)
- Owner_Type - First, Second, Third, or Fourth & Above
- Mileage - The standard mileage offered by the car company in kmpl or km/kg
- Engine - The displacement volume of the engine in CC.
- Power - The maximum power of the engine in bhp.
- Seats - The number of seats in the car.
- Price - The price of the car (target).

### Load Dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn

In [2]:
cars_df = pd.read_csv( "final_cars_maruti.csv" )

In [3]:
cars_df.sample(5)

,Location,Fuel_Type,Transmission,Owner_Type,Seats,Price,Age,Model,Mileage,Power,KM_Driven
313,Mumbai,Petrol,Automatic,First,5,3.80,6,celerio,23.10,67.04,53
882,Coimbatore,Petrol,Manual,First,5,3.41,12,swift,20.40,83.11,50
233,Coimbatore,Petrol,Manual,First,5,3.10,10,alto,19.70,46.30,48
805,Mumbai,Petrol,Manual,First,7,6.25,6,ertiga,16.02,93.70,69
848,Pune,Diesel,Manual,First,5,7.25,4,vitara,24.30,88.50,31


In [4]:
cars_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1010 entries, 0 to 1009
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Location      1010 non-null   object 
 1   Fuel_Type     1010 non-null   object 
 2   Transmission  1010 non-null   object 
 3   Owner_Type    1010 non-null   object 
 4   Seats         1010 non-null   int64  
 5   Price         1010 non-null   float64
 6   Age           1010 non-null   int64  
 7   Model         1010 non-null   object 
 8   Mileage       1010 non-null   float64
 9   Power         1010 non-null   float64
 10  KM_Driven     1010 non-null   int64  
dtypes: float64(3), int64(3), object(5)
memory usage: 86.9+ KB


In [5]:
cars_df.sample(10)

,Location,Fuel_Type,Transmission,Owner_Type,Seats,Price,Age,Model,Mileage,Power,KM_Driven
258,Hyderabad,Petrol,Manual,First,5,2.60,7,omni,14.00,35.00,34
57,Mumbai,Petrol,Automatic,First,5,6.20,4,swift,18.50,83.14,18
392,Hyderabad,Petrol,Manual,First,5,2.20,7,alto,20.92,67.10,90
54,Chennai,Diesel,Manual,First,5,7.15,4,vitara,24.30,88.50,45
861,Kolkata,Diesel,Manual,First,5,2.19,11,swift,19.30,73.90,34
52,Chennai,Petrol,Manual,Second,5,1.40,14,alto,19.70,46.30,56
421,Kochi,Diesel,Manual,First,5,9.38,3,vitara,24.30,88.50,45
145,Coimbatore,Diesel,Manual,First,5,6.46,6,swift,22.90,74.00,59
865,Delhi,Petrol,Manual,First,5,2.65,4,alto,24.70,47.30,35
903,Delhi,Petrol,Manual,First,5,6.60,2,baleno,21.40,83.10,13


### Feature Set Selection

In [6]:
cars_df.columns

Index(['Location', 'Fuel_Type', 'Transmission', 'Owner_Type', 'Seats', 'Price',
       'Age', 'Model', 'Mileage', 'Power', 'KM_Driven'],
      dtype='object')

In [7]:
x_features = ['KM_Driven', 'Fuel_Type', 'Age',
              'Transmission', 'Owner_Type', 'Model']

In [8]:
cat_vars = ['Fuel_Type', 'Transmission', 'Owner_Type', 'Model']

In [9]:
num_vars = list(set(x_features) - set(cat_vars))

In [10]:
num_vars

['KM_Driven', 'Age']

In [11]:
cars_df[x_features].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1010 entries, 0 to 1009
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   KM_Driven     1010 non-null   int64 
 1   Fuel_Type     1010 non-null   object
 2   Age           1010 non-null   int64 
 3   Transmission  1010 non-null   object
 4   Owner_Type    1010 non-null   object
 5   Model         1010 non-null   object
dtypes: int64(2), object(4)
memory usage: 47.5+ KB


### Need for Data Transformation

1. Data imputation for Seats Column
    - Mean imputation
2. Categorical Encoding for categorical columns
    - OHE Encoding
3. Data scaling
    - Standard scaling

### Setting X and y variables

In [12]:
X = cars_df[x_features]
y = cars_df['Price']

### Data Splitting

In [13]:
from sklearn.model_selection import train_test_split

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    train_size = 0.8,
                                                    random_state = 80)

In [15]:
X_train.shape

(808, 6)

In [16]:
X_test.shape

(202, 6)

### Data Imputation

In [17]:
from sklearn.impute import SimpleImputer

In [18]:
imputed_num_vars = ['Seats']

In [19]:
imputed_num_vars

['Seats']

In [20]:
non_imputed_num_vars = list(set(num_vars) - set(imputed_num_vars))

In [21]:
non_imputed_num_vars

['KM_Driven', 'Age']

In [22]:
mean_imputer = SimpleImputer(strategy='mean')

### Encode Categorical Variables

In [23]:
from sklearn.preprocessing import OneHotEncoder

In [24]:
ohe_encoder = OneHotEncoder(handle_unknown='ignore')

### Scaling numerical vars

In [25]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

### Creating Pipelines

In [26]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [27]:
imputed_num_transformer = Pipeline( steps = [
        ('imputation', mean_imputer),
        ('scaler', scaler)])

In [28]:
non_imputed_num_transformer = Pipeline( steps = [('scaler', scaler)])

In [29]:
cat_transformer = Pipeline( steps = [('ohencoder', ohe_encoder)])

In [30]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num_not_imputed', non_imputed_num_transformer, non_imputed_num_vars),
        ('catvars', cat_transformer, cat_vars)])

### KNN (K-Nearest Neighbor)


In [31]:
from sklearn.neighbors import KNeighborsRegressor

In [32]:
#knn = KNeighborsRegressor(n_neighbors=20)
knn = KNeighborsRegressor(n_neighbors=20, weights='distance')

In [33]:
knn_v1 = Pipeline(steps=[('preprocessor', preprocessor),
                          ('knn', knn)])

In [34]:
knn_v1.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num_not_imputed',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['KM_Driven', 'Age']),
                                                 ('catvars',
                                                  Pipeline(steps=[('ohencoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Fuel_Type', 'Transmission',
                                                   'Owner_Type', 'Model'])])),
                ('knn',
                 KNeighborsRegressor(n_neighbors=20, weights='distance'))])

In [35]:
from sklearn import set_config
set_config(display='diagram')

In [36]:
knn_v1

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num_not_imputed',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['KM_Driven', 'Age']),
                                                 ('catvars',
                                                  Pipeline(steps=[('ohencoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Fuel_Type', 'Transmission',
                                                   'Owner_Type', 'Model'])])),
                ('knn',
                 KNeighborsRegressor(n_neighbors=20, weights='distance'))])

### Predict on test set

In [37]:
y_pred = knn_v1.predict(X_test)

### K Fold Cross Validation

In [38]:
from sklearn.model_selection import cross_val_score

In [39]:
scores = cross_val_score( knn_v1,
                          X_train,
                          y_train,
                          cv = 10,
                          scoring = 'r2')

In [40]:
scores

array([0.83719904, 0.85503347, 0.83046009, 0.85093421, 0.84064089,
       0.8282538 , 0.79423513, 0.87336603, 0.87520529, 0.84934506])

In [41]:
scores.mean()

0.8434672999233992

In [42]:
scores.std()

0.02235405343569982

In [43]:
from joblib import dump

In [44]:
dump(knn_v1, "cars.pkl")

['cars.pkl']

In [45]:
import sklearn

In [46]:
sklearn.__version__

'1.6.1'